In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random
import pandas as pd
import pickle
import json

In [2]:
data = pd.read_csv("portas logicas/problemAND.csv")
data

,-1,-1.1,-1.2
0,-1,1,-1
1,1,-1,-1
2,1,1,1


In [3]:
x = data.iloc[:, :-1] 
y = data.iloc[:, -1] 
x_portas = np.array(x)
y_portas = np.array(y)
y_portas.shape

(3,)

In [4]:
x = np.load("CARACTERES COMPLETO/X.npy")
x = x.reshape(x.shape[0], -1)
Y = np.load("CARACTERES COMPLETO/Y_classe.npy")
Y.shape[1]
x.shape[1]

120

In [5]:
Y

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(1326, 26), dtype=float32)

In [6]:
class NnModel:
    def __init__(self, x: np.ndarray, y: np.ndarray, hidden_neurons: int = 64, output_neurons: int = 26):
        self.hidden_neurons = hidden_neurons
        self.x = x
        self.y = y
        self.output_neurons = output_neurons
        #Captura o número de colunas(variáveis) para o modelo
        self.input_neurons = self.x.shape[1]
        self.loss_history=[]

        #Incialização dos Pesos e Viés
        self.w1 = np.random.randn(self.input_neurons, self.hidden_neurons)
        self.b1 = np.zeros((1, self.hidden_neurons))
        self.w2 = np.random.randn(self.hidden_neurons, self.output_neurons)
        self.b2 = np.zeros((1, self.output_neurons))
        print(f'W1: {self.w1.shape}')
        print(f'B1: {self.b1.shape}')
        print(f'W2: {self.w2.shape}')
        print(f'B2: {self.b2.shape}')

        self.z1 = 0
        self.f1 = 0

    #Passo os valores de x como parâmetro, dado que quero utilizar a função do feedfoward para testar, enquanto o modelo self.x utilizo para treino
    def feedfoward(self, x: np.ndarray):
        #Equação da reta
        self.z1 = x.dot(self.w1) + self.b1

        #Função de Ativação dos neurônios de entrada para os neurônios ocultos
        self.f1 = np.tanh(self.z1)
        
        #Equação da reta (hidden layer)
        z2 = self.f1.dot(self.w2) + self.b2

        #Softmax para poder calcular as "probabilidades" de cada classe
        softmax = np.exp(z2)/np.sum(np.exp(z2), axis = 1, keepdims=True)
        return softmax

    def loss_function(self, softmax):
        #Aplicação da função de Cross Entropy para calcular o erro da rede neural
        losses = []
        for i in range(len(self.y)):
            y_pred = softmax[i, self.y[i]]
            losses.append(-np.log(y_pred))  # Evitar log(0)
        return np.mean(losses)

    def backpropation(self, softmax: np.ndarray, learning_rate: float):
        # DE -> derivada do erro
        # Dy_pred(\hat y) -> derivada do valor predito pela softmax na output layer
        # DZ2 -> derivada da equação de reta da hidden layer para a output layer
        # DW2 -> derivada dos pesos da hidden layer para a output layer
        # Df1 -> derivada da função de ativação da hidden layer (Tangente hiperbólica)
        # DZ1 -> derivada da equação de reta da input layer para a hidden layer
        # DW1 -> derivada dos pesos de entrada para a hidden layer
       
        delta2 = np.copy(softmax)
        delta2[range(self.y.shape[0]), self.y] -= 1
        dW2 = self.f1.T.dot(delta2)
        dB2 = np.sum(delta2, axis = 0, keepdims= True)

        delta1 = delta2.dot(self.w2.T) * (1 - np.power(np.tanh(self.z1) , 2))
        dW1 = self.x.T.dot(delta1)
        dB1 = np.sum(delta1, axis=0, keepdims=True)

        self.w1 +=  -learning_rate * dW1
        self.w2 += - learning_rate * dW2

        self.b1 += -learning_rate * dB1
        self.b2 += -learning_rate * dB2


    def fit(self, epochs, learning_rate: float):
        
        for epoch in range(epochs):
            output = self.feedfoward(self.x)
            loss = self.loss_function(output)
            self.backpropation(output, learning_rate)

            #Acurácia
            predictions = np.argmax(output, axis = 1)
            correct = (predictions == self.y).sum()
            accuracy = correct / len(self.y)
            
            self.loss_history.append(loss)
            if(epoch + 1) % (epochs//10) == 0:
                print(f'Epoch: [{epoch + 1} / {epochs}] Accuracy: [{accuracy:.3f}] Loss: [{loss:.3f}]')

        return predictions

In [7]:
y_indices = np.argmax(Y, axis=1)
y_indices

array([ 0,  1,  2, ..., 23, 24, 25], shape=(1326,))

In [ ]:
# ==============================
# PREPARAÇÃO DOS DADOS
# ==============================

y_indices = np.argmax(Y, axis=1)

np.random.seed(42)

n_samples = len(x)
indices = np.arange(n_samples)

np.random.shuffle(indices)

# 70 treino
# 10 validação
# 20 teste

train_end = int(0.7*n_samples)
val_end = int(0.8*n_samples)

train_indices = indices[:train_end]
val_indices = indices[train_end:val_end]
test_indices = indices[val_end:]


x_train = x[train_indices]
y_train = y_indices[train_indices]

x_val = x[val_indices]
y_val = y_indices[val_indices]

x_test = x[test_indices]
y_test = y_indices[test_indices]


# ==============================
# GRID SEARCH
# ==============================

hidden_neurons_options=[32, 50, 64, 85]

epochs_options=[100, 300, 500, 1000]

learning_rates=[0.001, 0.005, 0.01, 0.015, 0.02]

best_acc=0
best_model=None
best_params=None

best_initial_weights=None


for hidden_neurons in hidden_neurons_options:

    for epochs in epochs_options:

        for lr in learning_rates:

            print(f"\nTreinando:" f" Hidden={hidden_neurons}" f" Epochs={epochs}" f" LR={lr}")


            model = NnModel(x_train, y_train, hidden_neurons=hidden_neurons, output_neurons=26)

            # salva pesos iniciais
            initial_w1=model.w1.copy()
            initial_b1=model.b1.copy()

            initial_w2=model.w2.copy()
            initial_b2=model.b2.copy()

            predictions_train=model.fit(epochs=epochs, learning_rate=lr)

            predictions_val=np.argmax(model.feedfoward(x_val),axis=1)

            train_acc=(predictions_train==y_train).mean()

            val_acc=(predictions_val==y_val).mean()


            print(f"Treino:{train_acc:.4f}")

            print(f"Validação:{val_acc:.4f}")


            if val_acc > best_acc:

                best_acc=val_acc

                best_model=model

                best_params={"hidden":hidden_neurons, "epochs":epochs,"lr":lr}

                best_initial_weights={

                    "w1":initial_w1,
                    "b1":initial_b1,

                    "w2":initial_w2,
                    "b2":initial_b2
                }


print("\nMelhores parâmetros")

print(best_params)

print(
    f"Validação:{best_acc:.4f}"
)


# ==============================
# TESTE FINAL
# ==============================

predictions_test=np.argmax(best_model.feedfoward(x_test), axis=1)

test_acc=(predictions_test==y_test).mean()

print(f"\nTeste:{test_acc:.4f}")


# ==============================
# Arquivos exportados
# ==============================

print("\nExportando arquivos...")


# hiperparâmetros

hyperparameters={
    "input_neurons": x_train.shape[1], 
    "hidden_neurons": best_params["hidden"],
    "output_neurons":26,
    "epochs": best_params["epochs"], 
    "learning_rate": best_params["lr"], 
    "train": "70%", 
    "validation": "10%", 
    "test":"20%"
}


with open("hiperparametros_finais.json","w") as f: json.dump(hyperparameters, f,
        indent=4
    )


# pesos iniciais

pesos_iniciais={
    "w1":
    best_initial_weights["w1"].tolist(),

    "b1":
    best_initial_weights["b1"].tolist(),

    "w2":
    best_initial_weights["w2"].tolist(),

    "b2":
    best_initial_weights["b2"].tolist()
}


with open("pesos_iniciais.pkl","wb") as f:

    pickle.dump(
        pesos_iniciais,
        f
    )


# pesos finais

pesos_finais={

    "w1":
    best_model.w1.tolist(),

    "b1":
    best_model.b1.tolist(),

    "w2":
    best_model.w2.tolist(),

    "b2":
    best_model.b2.tolist()
}


with open("pesos_finais.pkl","wb") as f: pickle.dump(pesos_finais, f)


# erro treinamento

erro_df = pd.DataFrame({
    "epoca":np.arange(len(best_model.loss_history)),
    "erro": best_model.loss_history
})

erro_df.to_csv(
    "erro_treinamento.csv",
    index=False
)


# saídas do teste

probs = best_model.feedfoward(x_test)


resultado_df=pd.DataFrame({
    "classe_real":
    y_test,

    "classe_prevista":
    predictions_test
})


resultado_df.to_csv("saidas_teste.csv",index=False)


print("\nArquivos gerados:")

print("hiperparametros_finais.json")

print("pesos_iniciais.pkl")

print("pesos_finais.pkl")

print("erro_treinamento.csv")

print("saidas_teste.csv")


Treinando: Hidden=32 Epochs=100 LR=0.001
W1: (120, 32)
B1: (1, 32)
W2: (32, 26)
B2: (1, 26)
Epoch: [10 / 100] Accuracy: [0.166] Loss: [4.394]
Epoch: [20 / 100] Accuracy: [0.292] Loss: [2.991]
Epoch: [30 / 100] Accuracy: [0.397] Loss: [2.244]
Epoch: [40 / 100] Accuracy: [0.487] Loss: [1.805]
Epoch: [50 / 100] Accuracy: [0.566] Loss: [1.538]
Epoch: [60 / 100] Accuracy: [0.612] Loss: [1.342]
Epoch: [70 / 100] Accuracy: [0.651] Loss: [1.188]
Epoch: [80 / 100] Accuracy: [0.692] Loss: [1.060]
Epoch: [90 / 100] Accuracy: [0.728] Loss: [0.957]
Epoch: [100 / 100] Accuracy: [0.751] Loss: [0.876]
Treino:0.7511
Validação:0.5227

Treinando: Hidden=32 Epochs=100 LR=0.005
W1: (120, 32)
B1: (1, 32)
W2: (32, 26)
B2: (1, 26)
Epoch: [10 / 100] Accuracy: [0.387] Loss: [3.152]
Epoch: [20 / 100] Accuracy: [0.643] Loss: [1.299]
Epoch: [30 / 100] Accuracy: [0.706] Loss: [0.947]
Epoch: [40 / 100] Accuracy: [0.810] Loss: [0.600]
Epoch: [50 / 100] Accuracy: [0.841] Loss: [0.505]
Epoch: [60 / 100] Accuracy: [0.9

C:\Users\avera\AppData\Local\Temp\ipykernel_5728\2381500087.py:44: RuntimeWarning: divide by zero encountered in log
  losses.append(-np.log(y_pred))  # Evitar log(0)


Epoch: [30 / 100] Accuracy: [0.131] Loss: [inf]
Epoch: [40 / 100] Accuracy: [0.121] Loss: [174.869]
Epoch: [50 / 100] Accuracy: [0.086] Loss: [181.728]
Epoch: [60 / 100] Accuracy: [0.058] Loss: [274.859]
Epoch: [70 / 100] Accuracy: [0.071] Loss: [218.964]
Epoch: [80 / 100] Accuracy: [0.088] Loss: [189.596]
Epoch: [90 / 100] Accuracy: [0.122] Loss: [145.034]
Epoch: [100 / 100] Accuracy: [0.101] Loss: [156.240]
Treino:0.1013
Validação:0.0833

Treinando: Hidden=32 Epochs=300 LR=0.001
W1: (120, 32)
B1: (1, 32)
W2: (32, 26)
B2: (1, 26)
Epoch: [30 / 300] Accuracy: [0.490] Loss: [2.024]
Epoch: [60 / 300] Accuracy: [0.728] Loss: [1.051]
Epoch: [90 / 300] Accuracy: [0.818] Loss: [0.708]
Epoch: [120 / 300] Accuracy: [0.873] Loss: [0.515]
Epoch: [150 / 300] Accuracy: [0.906] Loss: [0.402]
Epoch: [180 / 300] Accuracy: [0.923] Loss: [0.334]
Epoch: [210 / 300] Accuracy: [0.942] Loss: [0.284]
Epoch: [240 / 300] Accuracy: [0.959] Loss: [0.242]
Epoch: [270 / 300] Accuracy: [0.973] Loss: [0.211]
Epoch: 

C:\Users\avera\AppData\Local\Temp\ipykernel_5728\2381500087.py:36: RuntimeWarning: overflow encountered in exp
  softmax = np.exp(z2)/np.sum(np.exp(z2), axis = 1, keepdims=True)
C:\Users\avera\AppData\Local\Temp\ipykernel_5728\2381500087.py:36: RuntimeWarning: invalid value encountered in divide
  softmax = np.exp(z2)/np.sum(np.exp(z2), axis = 1, keepdims=True)


Epoch: [30 / 100] Accuracy: [0.044] Loss: [nan]
Epoch: [40 / 100] Accuracy: [0.044] Loss: [nan]
Epoch: [50 / 100] Accuracy: [0.044] Loss: [nan]
Epoch: [60 / 100] Accuracy: [0.044] Loss: [nan]
Epoch: [70 / 100] Accuracy: [0.044] Loss: [nan]
Epoch: [80 / 100] Accuracy: [0.044] Loss: [nan]
Epoch: [90 / 100] Accuracy: [0.044] Loss: [nan]
Epoch: [100 / 100] Accuracy: [0.044] Loss: [nan]
Treino:0.0442
Validação:0.0152

Treinando: Hidden=50 Epochs=300 LR=0.001
W1: (120, 50)
B1: (1, 50)
W2: (50, 26)
B2: (1, 26)
Epoch: [30 / 300] Accuracy: [0.624] Loss: [1.631]
Epoch: [60 / 300] Accuracy: [0.856] Loss: [0.686]
Epoch: [90 / 300] Accuracy: [0.925] Loss: [0.391]
Epoch: [120 / 300] Accuracy: [0.950] Loss: [0.248]
Epoch: [150 / 300] Accuracy: [0.976] Loss: [0.171]
Epoch: [180 / 300] Accuracy: [0.989] Loss: [0.130]
Epoch: [210 / 300] Accuracy: [0.991] Loss: [0.105]
Epoch: [240 / 300] Accuracy: [0.994] Loss: [0.086]
Epoch: [270 / 300] Accuracy: [0.996] Loss: [0.070]
Epoch: [300 / 300] Accuracy: [0.997

## Backpropagation

$$\frac{\partial E}{\partial W1} = \frac{\partial E}{\partial \hat y} * \frac{\partial \hat y}{\partial Z2} * \frac{\partial Z2}{\partial W2} * \frac{\partial W2}{\partial f1} * \frac{\partial f1}{\partial Z1} * \frac{\partial Z1}{\partial W1} $$

$$ \delta_2 = \frac{\partial E}{\partial \hat y} * \frac{\partial \hat y}{\partial Z2}$$

Partindo de $\hat y$:
$$\hat y =\frac{e^{z_{21}}}{e^{z_{21}} + e^{z_{22}}}$$

Temos que $\frac{\partial \hat y}{\partial Z2}$ é:
$$u = e^{z_{21}}  \space \space \space \space \space \space v = e^{z_{21}} + e^{z_{22}}$$


$$\hat y = \frac{u}{v}$$
$$
\frac{\partial{\frac{u}{v}}}{\partial{Z2}}
=
\frac{u'v - uv'}{v^2} = \frac{
e^{z_{21}}(e^{z_{21}}+e^{z_{22}})
-
e^{z_{21}}e^{z_{21}}
}{
(e^{z_{21}}+e^{z_{22}})^2
}
=
\frac{
e^{z_{21}}e^{z_{22}}
}{
(e^{z_{21}}+e^{z_{22}})^2
}
$$
Agora calculamos:

$$
1-\hat y
$$

Substituindo:

$$
1-
\frac{e^{z_{21}}}
{e^{z_{21}}+e^{z_{22}}}
$$

Colocando no mesmo denominador:

$$
=
\frac{
e^{z_{21}}+e^{z_{22}}-e^{z_{21}}
}{
e^{z_{21}}+e^{z_{22}}
}
$$

Simplificando:

$$
1-\hat y
=
\frac{
e^{z_{22}}
}{
e^{z_{21}}+e^{z_{22}}
}
$$

Agora fazemos:

$$
\hat y(1-\hat y)
$$

Substituindo:

$$
=
\frac{e^{z_{21}}}
{e^{z_{21}}+e^{z_{22}}}
\cdot
\frac{e^{z_{22}}}
{e^{z_{21}}+e^{z_{22}}}
$$

Multiplicando:

$$
=
\frac{
e^{z_{21}}e^{z_{22}}
}{
(e^{z_{21}}+e^{z_{22}})^2
}
$$

Que é exatamente a derivada encontrada anteriormente. Assim temos que:
$$\frac{\partial \hat y}{\partial z_{21}} = \hat y(1-\hat y)$$


Agora vamos calcular a $\frac{\partial E}{\partial \hat y}$:
$$-Y'*\ln{\hat y} + -Y * \ln{\hat y}'$$

$$ \delta_2 = (0*\ln{\hat y} + (-Y * \frac{1}{\hat y})) * \frac{\partial \hat y}{\partial Z2} = (-Y * \frac{1}{\hat y}) * \hat y(1-\hat y) = -Y * (1- \hat y)  = \hat y - 1  $$
---
Cálculo das derivadas parciais da layer de entrada para a hidden layer

$$\frac{\partial E}{\partial W1} = \delta2 * W2 * \frac{\partial f1}{\partial Z1} * \frac{\partial Z1}{\partial W1}$$

$$\tanh = \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}}$$
$$\frac{\partial f1}{\partial Z1} =\frac{e^{z} - e^{-z}}{e^{z} + e^{-z}} = \frac{(e^{z} - e^{-z})(e^{z} + e^{-z}) - (e^{z} - e^{-z})(e^{z} + e^{-z})}{(e^{z} + e^{-z})²} = \frac{(e^{z} + e^{-z})² - (e^{z} - e^{-z})²}{(e^{z} + e^{-z})²} = 1 - \tanh²z $$
$$\frac{\partial Z1}{\partial W1} = X$$

$$\frac{\partial E}{\partial W1} = \delta2 * W2 * (1 - \tanh²z) *X$$


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import seaborn as sns

# ==============================
# ANÁLISE DE DESEMPENHO POR CLASSE
# ==============================

print("=" * 60)
print("RELATÓRIO DE CLASSIFICAÇÃO - DADOS DE TESTE")
print("=" * 60)
print("\n")
print(classification_report(y_test, predictions_test, target_names=[chr(65+i) for i in range(26)]))

# Cálculo de F1-score por classe
f1_scores = []
for class_idx in range(26):
    mask = y_test == class_idx
    if mask.sum() > 0:
        class_preds = predictions_test[mask]
        class_f1 = f1_score(y_test[mask], class_preds, average='binary' if len(np.unique(y_test[mask])) == 2 else 'macro', zero_division=0)
        f1_scores.append((class_idx, chr(65+class_idx), class_f1))

f1_df = pd.DataFrame(f1_scores, columns=['Classe', 'Letra', 'F1-Score'])
print("\n" + "=" * 60)
print("F1-SCORE POR CLASSE (ORDENADO)")
print("=" * 60)
print(f1_df.sort_values('F1-Score').to_string(index=False))

# Verificar especificamente a classe 0 (letra A)
print("\n" + "=" * 60)
print("ANÁLISE ESPECÍFICA DA LETRA A (Classe 0)")
print("=" * 60)
mask_A = y_test == 0
total_A = mask_A.sum()
correct_A = (predictions_test[mask_A] == 0).sum()
acc_A = correct_A / total_A if total_A > 0 else 0

print(f"Total de amostras da letra A nos testes: {total_A}")
print(f"Predições corretas: {correct_A}")
print(f"Acurácia para A: {acc_A:.2%}")

if total_A > 0:
    print(f"\nDistribuição de predições para amostras de A:")
    preds_for_A = predictions_test[mask_A]
    unique, counts = np.unique(preds_for_A, return_counts=True)
    for letter_pred, count in zip(unique, counts):
        print(f"  → Predita como '{chr(65+letter_pred)}': {count} ({count/total_A*100:.1f}%)")


In [ ]:
# ==============================
# MATRIZ DE CONFUSÃO
# ==============================

cm = confusion_matrix(y_test, predictions_test)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=False, cmap='Blues', cbar=True, ax=ax)
ax.set_xlabel('Predito')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusão - Teste')
ax.set_xticklabels([chr(65+i) for i in range(26)])
ax.set_yticklabels([chr(65+i) for i in range(26)])
plt.tight_layout()
plt.show()

# Analisar a linha da letra A especificamente
print("\n" + "=" * 60)
print("MATRIZ DE CONFUSÃO - LINHA A (linha 0)")
print("=" * 60)
print("Quando a verdade é A, o modelo previu:")
for i, count in enumerate(cm[0]):
    if count > 0:
        print(f"  {chr(65+i)}: {count} vezes")


In [ ]:
# ==============================
# ANÁLISE DE BALANCEAMENTO DE DADOS
# ==============================

print("\n" + "=" * 60)
print("DISTRIBUIÇÃO DE CLASSES NOS DADOS DE TREINAMENTO")
print("=" * 60)

class_counts_train = []
for i in range(26):
    count = (y_train == i).sum()
    class_counts_train.append((i, chr(65+i), count))

train_df = pd.DataFrame(class_counts_train, columns=['Idx', 'Letra', 'Qtd_Treino'])
print(train_df.sort_values('Qtd_Treino').to_string(index=False))

# Visualizar distribuição
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Treino
counts_train = [np.sum(y_train == i) for i in range(26)]
axes[0].bar([chr(65+i) for i in range(26)], counts_train, color='steelblue')
axes[0].set_title('Distribuição de Classes - Dados de Treinamento')
axes[0].set_ylabel('Número de amostras')
axes[0].tick_params(axis='x', rotation=45)

# Teste
counts_test = [np.sum(y_test == i) for i in range(26)]
axes[1].bar([chr(65+i) for i in range(26)], counts_test, color='coral')
axes[1].set_title('Distribuição de Classes - Dados de Teste')
axes[1].set_ylabel('Número de amostras')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\nClasse A (0) no treinamento: {counts_train[0]} amostras")
print(f"Classe A (0) no teste: {counts_test[0]} amostras")
print(f"\nMédia de amostras por classe (treino): {np.mean(counts_train):.1f}")
print(f"Classe A está {np.mean(counts_train)/counts_train[0]:.2f}x acima da média")


In [ ]:
# ==============================
# POSSÍVEIS SOLUÇÕES
# ==============================

print("\n" + "=" * 60)
print("RECOMENDAÇÕES PARA MELHORAR A PREDIÇÃO DE A")
print("=" * 60)

print("""
1. AUMENTAR EPOCH OU LEARNING RATE
   - O modelo pode não estar convergindo completamente
   - Tente adicionar mais épocas ou learning rates maiores

2. AUMENTAR NEURÔNIOS NA HIDDEN LAYER
   - Uma camada oculta com mais neurônios pode capturar 
     características mais complexas da letra A
   - Tente: 128, 256

3. USAR CLASS WEIGHTS (PONDERAÇÃO)
   - Se A tem poucas amostras, ponderar o loss para essa classe
   - Modificar a função de loss para penalizar mais erros em A

4. AUMENTO DE DADOS (DATA AUGMENTATION)
   - Se houver poucos exemplos de A, gerar variações

5. ADICIONAR MAIS CAMADAS OCULTAS
   - Usar arquitetura mais profunda (e.g., 2-3 camadas)

6. USAR OTIMIZADOR ADAPTATIVO
   - Substituir SGD por Adam ou RMSprop
""")

print("\nAnalisando o caso específico da classe A...")
if counts_train[0] < np.mean(counts_train):
    print(f"❌ PROBLEMA IDENTIFICADO: Classe A tem POUCOS dados ({counts_train[0]}) comparado à média ({np.mean(counts_train):.0f})")
else:
    print(f"✓ Classe A tem suficientes dados ({counts_train[0]})")

# Verificar padrão de confusão
confusoes_A = []
for i in range(26):
    if i != 0 and cm[0, i] > 0:
        confusoes_A.append((chr(65+i), cm[0, i]))

if confusoes_A:
    print(f"\n❌ A é frequentemente confundida com: {', '.join([f'{letter}({count})' for letter, count in sorted(confusoes_A, key=lambda x: x[1], reverse=True)[:3]])}")
